                # BetterTogether (Apple Silicon / MPS)

                Alternate prompt optimization and weight optimization, evaluate intermediate programs, and retain the best strategy stage.

                **Use it when:** You have both a useful prompt optimizer and a trainable local model, and want them to improve each other rather than run in isolation.

                **What compilation changes:** Prompt demonstrations first, then a Qwen LoRA adapter in the explicit `p -> w` candidate.

                Important configuration in this benchmark:

                - BootstrapFewShotWithRandomSearch (`p`) plus BootstrapFinetune (`w`)
- stock DSPy BetterTogether with explicit `p -> w` strategy
- MPS-backed Qwen2.5-0.5B-Instruct, 18 weight steps, seed 42
- preserve original, `p`, and `p -> w` candidates even when validation ties

                The 74-row dataset and pair-grouped train/validation/test membership are frozen.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import (
    artifact_paths,
    benchmark_snapshot,
    learned_program_preview,
    verify_prompt_artifact,
)

OPTIMIZER = 'better-together'
RUN_MODE = os.getenv("CHAPTER06_RUN", "inspect").lower()
print(f"mode={RUN_MODE!r}; optimizer={OPTIMIZER!r}")
print("safe default: inspect checked-in full-run artifacts; no API calls")

mode='inspect'; optimizer='better-together'
safe default: inspect checked-in full-run artifacts; no API calls


                ## Compile shape

                This is the essential DSPy call used by the shared runner (setup variables omitted):

                ```python
                optimizer = dspy.BetterTogether(
    metric=exact_match, p=prompt_optimizer, w=weight_optimizer,
)
optimized_detector = optimizer.compile(
    detector, trainset=trainset, teacher=local_teacher, valset=valset,
    strategy='p -> w', seed=42,
)
                ```

                `compile` returns a program. Calling that program on the untouched test examples is
                a separate phase; the notebook reports optimization cost/time separately from inference latency.

In [2]:
print(benchmark_snapshot(OPTIMIZER))
print()
print(artifact_paths(OPTIMIZER))

BetterTogether — frozen full-profile rerun
status: completed
task model: Qwen/Qwen2.5-0.5B-Instruct
test accuracy: 50.0%
uplift: +0.0 points vs its Qwen run baseline
optimization: $0.0000 and 4.6min
inference latency: mean 1.39s; p95 1.94s
reload checks: prompt=True; model=True; predictions=3/3 labels
comparison boundary: same frozen split, separate Qwen/MPS experiment; not Luna-comparable
complete run: chapter06/results/runs/full/better-together/20260715T034246.168669Z

Complete artifacts:
- candidate_programs: chapter06/results/runs/full/better-together/20260715T034246.168669Z/candidate_programs
- complete_output: chapter06/results/runs/full/better-together/20260715T034246.168669Z/complete_output.txt
- lm_history: chapter06/results/runs/full/better-together/20260715T034246.168669Z/lm_history.jsonl
- metrics: chapter06/results/runs/full/better-together/20260715T034246.168669Z/metrics.json
- optimizer_state: chapter06/results/runs/full/better-together/20260715T034246.168669Z/optimizer_

## Read the result

DSPy retained the original program when all validation candidates tied. That is not a failed run: inspect `candidate_programs/` to see the completed prompt-only and trained `p -> w` alternatives and the adapter that was deliberately preserved.

The next cell shows a bounded readable preview. The complete, lossless prompt and
serialized program remain at the paths printed above.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 0

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt_state_equal': True, 'mismatches': []}


## Run it yourself (explicit opt-in)

The default `inspect` mode is offline and free. For a live run, first install from
the repository root with `python -m pip install -r requirements.txt`, configure
`OPENAI_API_KEY`, and restart the kernel.

- Bounded code-path check: launch Jupyter with `CHAPTER06_RUN=smoke`.
- Complete frozen-split rerun: launch Jupyter with `CHAPTER06_RUN=full`.

A full run is intentionally not triggered by opening or choosing “Run All”: it can
take minutes or incur model charges. The weight optimizers download and train a
small Qwen model locally through MPS/CPU and require the optional training stack. Live
artifacts are written to `chapter06/results/runs/<profile>/<optimizer>/<run-id>/`.
Rebuild the comparison afterward with `python -m chapter06.summarize_optimizer_results`.

In [4]:
if RUN_MODE == "inspect":
    print("Live run skipped. Set CHAPTER06_RUN=smoke or CHAPTER06_RUN=full explicitly.")
elif RUN_MODE in {"smoke", "full"}:
    from chapter06.optimizer_experiment import run_experiment

    live_result = run_experiment(OPTIMIZER, profile_name=RUN_MODE)
    print(live_result)
else:
    raise ValueError("CHAPTER06_RUN must be inspect, smoke, or full")

Live run skipped. Set CHAPTER06_RUN=smoke or CHAPTER06_RUN=full explicitly.
